# Notebook 02: Preprocessing and ML Data Preparation

This notebook prepares the merged Week 5 dataset for modelling, It:

1. Loads the merged Banner and Moodle dataset from Notebook 01.
2. Converts the required count, GPA, activity, and engagement fields.
3. Corrects the known credits_earned_ratio source artifact.
4. Defines and validates the early prediction feature set.
5. Excludes identifiers, target leakage, and gender from model predictors.
6. Creates one stratified 80/20 train and test split.
7. Fits preprocessing on the training set only.
8. Saves the data and preprocessing objects required by later notebooks.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import joblib

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "academic_risk_label"

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

INPUT_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "merged_banner_moodle_with_target.pkl"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "ml_ready"
)

RESULTS_DIR = PROJECT_DIR / "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

Project directory: c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks


## 2. Load and validate the merged dataset

In [2]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

data = pd.read_pickle(INPUT_FILE).reset_index(drop=True)

if TARGET not in data.columns:
    raise KeyError(
        f"Target column not found: {TARGET}"
    )

if data[TARGET].isna().any():
    raise ValueError(
        "Missing target labels found. "
        "Notebook 01 should remove unavailable targets."
    )

data[TARGET] = data[TARGET].astype(int)

print("Loaded:", INPUT_FILE)
print("Shape:", data.shape)

display(
    data[TARGET]
    .value_counts()
    .sort_index()
    .rename(index={0: "Not at risk", 1: "At risk"})
    .rename_axis("risk_class")
    .to_frame("students")
)

Loaded: c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks\data\processed\merged_banner_moodle_with_target.pkl
Shape: (3760, 38)


,students
risk_class,
Not at risk,2827
At risk,933


## 3. Convert source fields required for modelling

The following deterministic cleaning and conversions are applied:

- count fields containing `5+` are converted to numeric value `5`
- `active_days = 3 or fewer` is represented as `3`
- previous GPA and previous CGPA bands are converted to ordered values from 0 to 5
- Moodle engagement categories are converted to ordered values from 0 to 4
- Correct `credits_earned_ratio` values above 1.0 which known source data artifact and are capped at 1.0

In [3]:
CAPPED_COUNT_COLUMNS = [
    "repeated_course_count_current",
    "previous_failed_course_count",
    "previous_withdrawn_course_count",
    "previous_repeated_course_count",
]

for column in CAPPED_COUNT_COLUMNS:
    data[column] = pd.to_numeric(
        data[column]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("5+", "5", regex=False),
        errors="coerce",
    )


def normalise_category(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("_", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )


GPA_RANGE_MAP = {
    "0.00 TO <1.00": 0,
    "1.00 TO <2.00": 1,
    "2.00 TO <2.25": 2,
    "2.25 TO <2.50": 3,
    "2.50 TO <3.00": 4,
    "3.00 TO 4.00": 5,
}

ENGAGEMENT_MAP = {
    "VERY LOW": 0,
    "LOW": 1,
    "MODERATE": 2,
    "GOOD": 3,
    "HIGH": 4,
}

data["previous_term_gpa_ordinal"] = (
    normalise_category(data["previous_term_gpa_range"])
    .map(GPA_RANGE_MAP)
)

data["previous_cgpa_ordinal"] = (
    normalise_category(data["previous_cgpa_range"])
    .map(GPA_RANGE_MAP)
)

data["learning_material_events_ordinal"] = (
    normalise_category(data["learning_material_events"])
    .map(ENGAGEMENT_MAP)
)

data["assessment_interaction_events_ordinal"] = (
    normalise_category(data["assessment_interaction_events"])
    .map(ENGAGEMENT_MAP)
)

active_days_clean = (
    data["active_days"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"3 or fewer": "3"})
)

data["active_days_numeric"] = pd.to_numeric(
    active_days_clean,
    errors="raise",
)

# Known source artifact in historical cumulative credits earned ratio
data["credits_earned_ratio"] = pd.to_numeric(
    data["credits_earned_ratio"],
    errors="coerce",
)

data["credits_earned_ratio"] = (
    data["credits_earned_ratio"]
    .clip(upper=1.0)
)

print(
    "Maximum credits_earned_ratio:",
    data["credits_earned_ratio"].max(),
)
print("Required modelling fields converted.")

Maximum credits_earned_ratio: 1.0
Required modelling fields converted.


## 4. Define the model feature set

The selected predictors use only information available before or during the first five weeks.

Gender is retained separately for fairness evaluation and is never used as a model predictor.

In [4]:
NUMERIC_FEATURES = [
    # Banner workload and academic history
    "registered_course_count",
    "registered_credits",
    "repeated_course_count_current",
    "previous_failed_course_count",
    "previous_withdrawn_course_count",
    "previous_repeated_course_count",
    "credits_earned_ratio",
    # Converted fields
    "previous_term_gpa_ordinal",
    "previous_cgpa_ordinal",

    # Week 1 to 5 attendance
    "attendance_rate",
    "absence_count",

    # Week 1 to 5 Moodle
    "enrolled_course_count",
    "accessed_course_count",
    "course_access_rate",
    "total_course_event_clicks",
    "active_days_rate",
    "zero_activity_days",
    "largest_inactivity_days",
    # Converted Moodle activity features
    "active_days_numeric",
    "learning_material_events_ordinal",
    "assessment_interaction_events_ordinal",
]


CATEGORICAL_FEATURES = [
    "programme_or_school",
    "year_level",
    "previous_academic_standing",
]

SELECTED_FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

FAIRNESS_COLUMN = "gender"

LEAKAGE_COLUMNS = {
    "end_term_gpa",
    "end_cgpa",
    "end_term_gpa_range",
    "end_cgpa_range",
    "earned_credits_this_semester",
    "credit_deficit",
    "risk_source",
    TARGET,
}

REDUNDANT_OR_REPLACED_COLUMNS = {
    "active_days",
    "previous_term_gpa",
    "previous_term_gpa_range",
    "previous_cgpa",
    "previous_cgpa_range",
    "learning_material_events",
    "assessment_interaction_events",
}

missing_features = [
    feature
    for feature in SELECTED_FEATURES
    if feature not in data.columns
]

if missing_features:
    raise KeyError(
        "Expected modelling features are missing: "
        f"{missing_features}"
    )

unexpected_leakage = sorted(
    set(SELECTED_FEATURES)
    & LEAKAGE_COLUMNS
)

if unexpected_leakage:
    raise ValueError(
        "Leakage found in selected predictors: "
        f"{unexpected_leakage}"
    )

if FAIRNESS_COLUMN not in data.columns:
    raise KeyError(
        f"Fairness column not found: {FAIRNESS_COLUMN}"
    )

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total raw model features:", len(SELECTED_FEATURES))

display(
    pd.DataFrame({
        "feature": SELECTED_FEATURES,
        "type": (
            ["Numeric"] * len(NUMERIC_FEATURES)
            + ["Categorical"] * len(CATEGORICAL_FEATURES)
        ),
    })
)

Numeric features: 21
Categorical features: 3
Total raw model features: 24


,feature,type
0,registered_course_count,Numeric
1,registered_credits,Numeric
2,repeated_course_count_current,Numeric
3,previous_failed_course_count,Numeric
4,previous_withdrawn_course_count,Numeric
5,previous_repeated_course_count,Numeric
6,credits_earned_ratio,Numeric
7,previous_term_gpa_ordinal,Numeric
8,previous_cgpa_ordinal,Numeric
9,attendance_rate,Numeric


## 5. Data dictionary

The data dictionary records each field's modelling role, data type, missingness, unique values, and either its numeric range or its available categories.

Identifier values are hidden.

In [5]:
def modelling_role(column):
    if column == "student_key":
        return "Identifier"
    if column == FAIRNESS_COLUMN:
        return "Fairness only"
    if column in LEAKAGE_COLUMNS:
        return "Leakage or target construction"
    if column in NUMERIC_FEATURES:
        return "Selected numeric predictor"
    if column in CATEGORICAL_FEATURES:
        return "Selected categorical predictor"
    if column in REDUNDANT_OR_REPLACED_COLUMNS:
        return "Excluded redundant or replaced source"
    return "Not used for modelling"


def range_or_categories(column):
    series = data[column]

    if column == "student_key":
        return "Values hidden for privacy"

    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(
            series,
            errors="coerce",
        )

        if numeric.notna().any():
            return (
                f"Min: {numeric.min():.3f}, "
                f"Max: {numeric.max():.3f}"
            )

        return "No valid numeric values"

    cleaned = (
        series
        .astype("string")
        .str.strip()
        .dropna()
    )

    categories = cleaned.unique().tolist()

    if len(categories) <= 20:
        return " | ".join(
            map(str, categories)
        )

    return (
        "Sample: "
        + " | ".join(
            map(str, categories[:5])
        )
    )


data_dictionary = pd.DataFrame({
    "column": data.columns,
    "dtype": [
        str(data[column].dtype)
        for column in data.columns
    ],
    "modelling_role": [
        modelling_role(column)
        for column in data.columns
    ],
    "missing_count": [
        int(data[column].isna().sum())
        for column in data.columns
    ],
    "missing_percentage": [
        round(
            data[column].isna().mean() * 100,
            2,
        )
        for column in data.columns
    ],
    "unique_count": [
        int(data[column].nunique(dropna=True))
        for column in data.columns
    ],
    "range_or_categories": [
        range_or_categories(column)
        for column in data.columns
    ],
})

display(data_dictionary)

DATA_DICTIONARY_FILE = (
    RESULTS_DIR
    / "data_dictionary.xlsx"
)

data_dictionary.to_excel(
    DATA_DICTIONARY_FILE,
    index=False,
)

print(
    "Saved data dictionary:",
    DATA_DICTIONARY_FILE,
)

,column,dtype,modelling_role,missing_count,missing_percentage,unique_count,range_or_categories
0,student_key,string,Identifier,0,0.00,3760,Values hidden for privacy
1,semester_code_banner,int64,Not used for modelling,0,0.00,1,"Min: 202502.000, Max: 202502.000"
2,programme_or_school,str,Selected categorical predictor,0,0.00,6,Bachelor of Business | Bachelor of Engineering...
3,year_level,int64,Selected categorical predictor,0,0.00,4,"Min: 1.000, Max: 4.000"
4,gender,str,Fairness only,0,0.00,3,Female | Male | Not Disclosed
5,registered_course_count,int64,Selected numeric predictor,0,0.00,9,"Min: 1.000, Max: 9.000"
6,registered_credits,int64,Selected numeric predictor,0,0.00,23,"Min: 10.000, Max: 125.000"
7,repeated_course_count_current,Int64,Selected numeric predictor,0,0.00,6,"Min: 0.000, Max: 5.000"
8,previous_term_gpa,str,Excluded redundant or replaced source,303,8.06,6,High | Good | Low | Near Threshold | Very Low ...
9,previous_term_gpa_range,str,Excluded redundant or replaced source,303,8.06,6,3.00 to 4.00 | 2.50 to <3.00 | 1.00 to <2.00 |...


Saved data dictionary: c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks\results\data_dictionary.xlsx


## 6. Create X, y, and fairness data

Gender is kept completely separate from model inputs so it can later be joined to held out predictions for fairness evaluation.

In [6]:
X = data[SELECTED_FEATURES].copy()
y = data[TARGET].copy()

fairness_data = data[
    [FAIRNESS_COLUMN]
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fairness data shape:", fairness_data.shape)

missing_selected = (
    X.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_selected["missing_percentage"] = (
    missing_selected["missing_count"]
    / len(X)
    * 100
).round(2)

display(
    missing_selected[
        missing_selected["missing_count"] > 0
    ]
)

X shape: (3760, 24)
y shape: (3760,)
Fairness data shape: (3760, 1)


,missing_count,missing_percentage
previous_term_gpa_ordinal,303,8.06
previous_cgpa_ordinal,295,7.85
previous_academic_standing,294,7.82


## 7. Create one stratified 80/20 train and test split

The split is performed before fitting the preprocessing pipeline.

The held out test set remains untouched during model tuning and model selection.

In [7]:
indices = np.arange(len(data))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

fairness_test = (
    fairness_data
    .iloc[test_indices]
    .copy()
)

split_summary = pd.DataFrame({
    "split": ["Train", "Test"],
    "rows": [
        len(X_train),
        len(X_test),
    ],
    "at_risk_students": [
        int(y_train.sum()),
        int(y_test.sum()),
    ],
    "at_risk_percentage": [
        round(float(y_train.mean() * 100), 2),
        round(float(y_test.mean() * 100), 2),
    ],
})

display(split_summary)

,split,rows,at_risk_students,at_risk_percentage
0,Train,3008,746,24.80
1,Test,752,187,24.87


## 8. Build and fit the baseline preprocessing pipeline

Numeric predictors:

- median imputation
- standard scaling

Categorical predictors:

- most frequent imputation
- one hot encoding

The preprocessor is fitted on the training set only and then applied to the held out test set.


In [8]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train_prepared_array = (
    preprocessor.fit_transform(X_train)
)

X_test_prepared_array = (
    preprocessor.transform(X_test)
)

feature_names = (
    preprocessor.get_feature_names_out()
)

X_train_prepared = pd.DataFrame(
    X_train_prepared_array,
    columns=feature_names,
    index=X_train.index,
)

X_test_prepared = pd.DataFrame(
    X_test_prepared_array,
    columns=feature_names,
    index=X_test.index,
)

print(
    "Prepared training shape:",
    X_train_prepared.shape,
)

print(
    "Prepared testing shape:",
    X_test_prepared.shape,
)

Prepared training shape: (3008, 36)
Prepared testing shape: (752, 36)


## 9. Validate the prepared data

In [9]:
validation = pd.DataFrame({
    "dataset": ["Train", "Test"],
    "rows": [
        len(X_train_prepared),
        len(X_test_prepared),
    ],
    "prepared_features": [
        X_train_prepared.shape[1],
        X_test_prepared.shape[1],
    ],
    "missing_values": [
        int(
            X_train_prepared
            .isna()
            .sum()
            .sum()
        ),
        int(
            X_test_prepared
            .isna()
            .sum()
            .sum()
        ),
    ],
})

display(validation)

assert (
    X_train_prepared.shape[1]
    == X_test_prepared.shape[1]
)

assert (
    X_train_prepared
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    X_test_prepared
    .isna()
    .sum()
    .sum()
    == 0
)

assert list(
    X_train_prepared.columns
) == list(
    X_test_prepared.columns
)

print("Prepared data validation passed.")

,dataset,rows,prepared_features,missing_values
0,Train,3008,36,0
1,Test,752,36,0


Prepared data validation passed.


In [10]:
print("Raw predictors:", len(SELECTED_FEATURES))
print("Prepared predictors after one hot encoding:", len(feature_names))

Raw predictors: 24
Prepared predictors after one hot encoding: 36


## 10. Save required outputs

The raw split is retained because later cross validation and tuning must refit preprocessing inside each training fold.

The prepared split is retained for baseline modelling and explainability workflows.

`fairness_test.pkl` contains gender for the held out test students only and is not part of model training.

In [11]:
X_train.to_pickle(
    OUTPUT_DIR / "X_train_raw.pkl"
)

X_test.to_pickle(
    OUTPUT_DIR / "X_test_raw.pkl"
)

X_train_prepared.to_pickle(
    OUTPUT_DIR / "X_train_prepared.pkl"
)

X_test_prepared.to_pickle(
    OUTPUT_DIR / "X_test_prepared.pkl"
)

y_train.to_pickle(
    OUTPUT_DIR / "y_train.pkl"
)

y_test.to_pickle(
    OUTPUT_DIR / "y_test.pkl"
)

fairness_test.to_pickle(
    OUTPUT_DIR / "fairness_test.pkl"
)

joblib.dump(
    preprocessor,
    OUTPUT_DIR / "preprocessor.joblib",
)

print("Saved ML preparation outputs to:")
print(OUTPUT_DIR)

print("\nSaved files:")
for file_name in [
    "X_train_raw.pkl",
    "X_test_raw.pkl",
    "X_train_prepared.pkl",
    "X_test_prepared.pkl",
    "y_train.pkl",
    "y_test.pkl",
    "fairness_test.pkl",
    "preprocessor.joblib",
]:
    print("-", file_name)

Saved ML preparation outputs to:
c:\Users\israa.tajaldin\OneDrive - Bahrain Polytechnic\Group Group IT9099 - Israa Tajaldin - General\04_Project_Notebooks\data\processed\ml_ready

Saved files:
- X_train_raw.pkl
- X_test_raw.pkl
- X_train_prepared.pkl
- X_test_prepared.pkl
- y_train.pkl
- y_test.pkl
- fairness_test.pkl
- preprocessor.joblib


## Notebook 02 complete
